# Module 2.2: Viterbi Algorithm Implementation

## Learning Objectives
By completing this notebook, you will:
1. Understand the difference between Viterbi decoding and marginal probability maximization
2. Implement the Viterbi algorithm for finding the most likely state sequence
3. Apply log-space computation to prevent numerical underflow
4. Extend Viterbi to Gaussian HMMs for continuous observations
5. Use Viterbi for market regime detection and compare with forward-backward

## Prerequisites
- Forward-backward algorithm (Module 2.1)
- Dynamic programming fundamentals
- Understanding of argmax vs. marginalization

## Estimated Time: 50 minutes

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, List
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. The Decoding Problem

**Goal:** Given observations O and model λ, find the most likely state sequence S*.

$$S^* = \arg\max_S P(S | O, \lambda)$$

### Why Not Just Take Marginal Maxima?

Taking `argmax γₜ(i)` at each time independently can give an **impossible** sequence with zero probability!

**Example:**
```
Transition matrix (deterministic alternation):
A = [[0.0, 1.0],   # State 0 MUST go to state 1
     [1.0, 0.0]]   # State 1 MUST go to state 0
```

Any sequence like [0, 0, ...] or [1, 1, ...] has **zero probability**, but marginal maximization might produce it.

In [ ]:
# Purpose: Demonstrate why marginal maximization can fail
# Key Concept: Most likely state sequence ≠ sequence of most likely states

def demonstrate_marginal_vs_viterbi():
    """
    Show a case where marginal and Viterbi differ.
    """
    # HMM with strong self-persistence
    pi = np.array([0.5, 0.5])
    A = np.array([
        [0.95, 0.05],  # State 0 very persistent
        [0.05, 0.95]   # State 1 very persistent
    ])
    
    # Emissions favor different states
    B = np.array([
        [0.9, 0.1],  # State 0 emits symbol 0 with high prob
        [0.1, 0.9]   # State 1 emits symbol 1 with high prob
    ])
    
    # Observation: mostly 0s with one 1 in the middle
    obs = np.array([0, 0, 0, 1, 0, 0, 0])
    
    print("Demonstration: Marginal vs. Viterbi Decoding")
    print("="*60)
    print(f"Observations: {obs}")
    print(f"\nTransition matrix (high self-persistence):")
    print(A)
    
    return pi, A, B, obs

pi_demo, A_demo, B_demo, obs_demo = demonstrate_marginal_vs_viterbi()

## 2. The Viterbi Algorithm

### Viterbi Variable Definition

$$\delta_t(i) = \max_{s_1,...,s_{t-1}} P(s_1, ..., s_{t-1}, s_t=i, o_1, ..., o_t | \lambda)$$

Probability of the **most likely path** ending in state i at time t.

### Recursion

**Initialization:**
$$\delta_1(i) = \pi_i \cdot b_i(o_1)$$

**Induction:**
$$\delta_t(j) = \max_i [\delta_{t-1}(i) \cdot a_{ij}] \cdot b_j(o_t)$$

**Backtracking pointer:**
$$\psi_t(j) = \arg\max_i [\delta_{t-1}(i) \cdot a_{ij}]$$

### Key Difference from Forward

- **Forward**: Sum over previous states (Σ)
- **Viterbi**: Max over previous states (max)

In [ ]:
# Purpose: Implement Viterbi algorithm for discrete HMM
# Key Concept: Dynamic programming with max instead of sum

def viterbi_algorithm(
    observations: np.ndarray,
    pi: np.ndarray,
    A: np.ndarray,
    B: np.ndarray
) -> Tuple[List[int], float, np.ndarray, np.ndarray]:
    """
    Viterbi algorithm for finding most likely state sequence.
    
    Parameters
    ----------
    observations : ndarray (T,)
        Observation sequence (discrete indices)
    pi : ndarray (K,)
        Initial state distribution
    A : ndarray (K, K)
        Transition matrix
    B : ndarray (K, M)
        Emission matrix
    
    Returns
    -------
    best_path : list of int
        Most likely state sequence
    best_prob : float
        Probability of best path
    delta : ndarray (T, K)
        Viterbi probabilities at each time
    psi : ndarray (T, K)
        Backtrack pointers
    """
    T = len(observations)
    K = len(pi)
    
    # Step 1: Initialize storage
    delta = np.zeros((T, K))
    psi = np.zeros((T, K), dtype=int)
    
    # Step 2: Initialization (t=0)
    # δ₁(i) = πᵢ × bᵢ(o₁)
    delta[0] = pi * B[:, observations[0]]
    psi[0] = 0  # No previous state
    
    # Step 3: Forward pass - find max probability paths
    for t in range(1, T):
        for j in range(K):
            # Find best previous state
            # δₜ(j) = maxᵢ [δₜ₋₁(i) × aᵢⱼ] × bⱼ(oₜ)
            probs = delta[t-1] * A[:, j]
            psi[t, j] = np.argmax(probs)
            delta[t, j] = probs[psi[t, j]] * B[j, observations[t]]
    
    # Step 4: Backtrack to find best path
    best_path = [0] * T
    
    # Start from best final state
    best_path[T-1] = np.argmax(delta[T-1])
    best_prob = delta[T-1, best_path[T-1]]
    
    # Follow backtrack pointers
    for t in range(T-2, -1, -1):
        best_path[t] = psi[t+1, best_path[t+1]]
    
    return best_path, best_prob, delta, psi

# Test on weather example
pi = np.array([0.6, 0.4])
A = np.array([[0.7, 0.3], [0.4, 0.6]])
B = np.array([[0.5, 0.5], [0.1, 0.9]])
observations = np.array([0, 0, 1, 1, 0])

best_path, best_prob, delta, psi = viterbi_algorithm(observations, pi, A, B)

print("Viterbi Algorithm Results:")
print("="*60)
print(f"Observations: {observations}")
print(f"\nBest state path: {best_path}")
print(f"Path probability: {best_prob:.8e}")
print(f"\nDelta (Viterbi probabilities):")
print(delta)
print(f"\nPsi (backtrack pointers):")
print(psi)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Top: Delta values
ax1 = axes[0]
ax1.plot(delta[:, 0], 'o-', label='δ(State 0)', linewidth=2, markersize=8)
ax1.plot(delta[:, 1], 's-', label='δ(State 1)', linewidth=2, markersize=8)
ax1.set_ylabel('Viterbi Probability', fontsize=12)
ax1.set_title('Viterbi Variables Over Time', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bottom: Best path
ax2 = axes[1]
ax2.fill_between(range(T), best_path, alpha=0.6, step='mid')
ax2.set_xlabel('Time Step', fontsize=12)
ax2.set_ylabel('State', fontsize=12)
ax2.set_title('Most Likely State Sequence (Viterbi Path)', fontsize=14)
ax2.set_yticks([0, 1])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Log-Space Viterbi

For numerical stability, work in log space:

$$\log \delta_t(j) = \max_i [\log \delta_{t-1}(i) + \log a_{ij}] + \log b_j(o_t)$$

This prevents underflow and is essential for long sequences.

In [ ]:
# Purpose: Implement log-space Viterbi for numerical stability
# Key Concept: Work in log space to prevent underflow

def viterbi_log(
    observations: np.ndarray,
    log_pi: np.ndarray,
    log_A: np.ndarray,
    log_B: np.ndarray
) -> Tuple[List[int], float, np.ndarray]:
    """
    Viterbi algorithm in log-space.
    
    Parameters
    ----------
    log_pi : ndarray (K,)
        Log initial probabilities
    log_A : ndarray (K, K)
        Log transition probabilities
    log_B : ndarray (K, M)
        Log emission probabilities
    
    Returns
    -------
    best_path : list of int
        Most likely state sequence
    log_prob : float
        Log probability of best path
    log_delta : ndarray (T, K)
        Log Viterbi probabilities
    """
    T = len(observations)
    K = len(log_pi)
    
    log_delta = np.zeros((T, K))
    psi = np.zeros((T, K), dtype=int)
    
    # Step 1: Initialization (in log space)
    log_delta[0] = log_pi + log_B[:, observations[0]]
    
    # Step 2: Induction
    for t in range(1, T):
        for j in range(K):
            # log δₜ(j) = maxᵢ [log δₜ₋₁(i) + log aᵢⱼ] + log bⱼ(oₜ)
            probs = log_delta[t-1] + log_A[:, j]
            psi[t, j] = np.argmax(probs)
            log_delta[t, j] = probs[psi[t, j]] + log_B[j, observations[t]]
    
    # Step 3: Backtrack
    best_path = [0] * T
    best_path[T-1] = np.argmax(log_delta[T-1])
    log_prob = log_delta[T-1, best_path[T-1]]
    
    for t in range(T-2, -1, -1):
        best_path[t] = psi[t+1, best_path[t+1]]
    
    return best_path, log_prob, log_delta

# Convert to log space and test
log_pi = np.log(pi + 1e-10)
log_A = np.log(A + 1e-10)
log_B = np.log(B + 1e-10)

best_path_log, log_prob, log_delta = viterbi_log(observations, log_pi, log_A, log_B)

print("Log-Space Viterbi:")
print("="*60)
print(f"Best path: {best_path_log}")
print(f"Log probability: {log_prob:.6f}")
print(f"Probability: {np.exp(log_prob):.8e}")
print(f"\nVerification - same as regular Viterbi: {best_path_log == best_path}")

## 4. Viterbi for Gaussian HMMs

For continuous observations, we compute emission probabilities using Gaussian densities instead of lookup tables.

In [ ]:
# Purpose: Implement Viterbi for Gaussian HMM (continuous observations)
# Key Concept: Replace discrete emissions with Gaussian densities

def viterbi_gaussian(
    observations: np.ndarray,
    pi: np.ndarray,
    A: np.ndarray,
    means: np.ndarray,
    stds: np.ndarray
) -> Tuple[List[int], float]:
    """
    Viterbi algorithm for Gaussian HMM.
    
    Parameters
    ----------
    observations : ndarray (T,)
        Continuous observation sequence
    means : ndarray (K,)
        Mean of each state's Gaussian emission
    stds : ndarray (K,)
        Standard deviation of each state's Gaussian emission
    
    Returns
    -------
    best_path : list of int
        Most likely state sequence
    log_prob : float
        Log probability of best path
    """
    T = len(observations)
    K = len(pi)
    
    # Step 1: Compute log emission probabilities for all observations
    # log_B[t, k] = log P(obs_t | state_k)
    log_B = np.zeros((T, K))
    for t in range(T):
        for k in range(K):
            log_B[t, k] = stats.norm.logpdf(
                observations[t],
                loc=means[k],
                scale=stds[k]
            )
    
    # Step 2: Convert parameters to log space
    log_pi = np.log(pi + 1e-10)
    log_A = np.log(A + 1e-10)
    
    # Step 3: Run Viterbi in log space
    log_delta = np.zeros((T, K))
    psi = np.zeros((T, K), dtype=int)
    
    # Initialization
    log_delta[0] = log_pi + log_B[0]
    
    # Induction
    for t in range(1, T):
        for j in range(K):
            probs = log_delta[t-1] + log_A[:, j]
            psi[t, j] = np.argmax(probs)
            log_delta[t, j] = probs[psi[t, j]] + log_B[t, j]
    
    # Backtrack
    best_path = [0] * T
    best_path[T-1] = np.argmax(log_delta[T-1])
    log_prob = log_delta[T-1, best_path[T-1]]
    
    for t in range(T-2, -1, -1):
        best_path[t] = psi[t+1, best_path[t+1]]
    
    return best_path, log_prob

# Example: Market regime detection
np.random.seed(42)

# True parameters
true_pi = np.array([0.5, 0.5])
true_A = np.array([[0.9, 0.1], [0.1, 0.9]])
true_means = np.array([0.001, -0.002])  # Bull: +0.1%, Bear: -0.2%
true_stds = np.array([0.01, 0.025])     # Bull: 1% vol, Bear: 2.5% vol

# Generate synthetic data
T = 100
true_states = [0]  # Start in Bull
for _ in range(T-1):
    true_states.append(np.random.choice(2, p=true_A[true_states[-1]]))

observations_cont = np.array([
    np.random.normal(true_means[s], true_stds[s]) for s in true_states
])

# Decode with Viterbi
decoded_path, log_prob = viterbi_gaussian(
    observations_cont, true_pi, true_A, true_means, true_stds
)

# Compute accuracy
accuracy = np.mean(np.array(decoded_path) == np.array(true_states))

print("Gaussian HMM Viterbi Results:")
print("="*60)
print(f"Sequence length: {T}")
print(f"Decoding accuracy: {accuracy:.1%}")
print(f"Log probability: {log_prob:.2f}")

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Observations
ax1 = axes[0]
ax1.plot(observations_cont, 'k-', alpha=0.7)
ax1.set_ylabel('Return', fontsize=12)
ax1.set_title('Market Returns (Observations)', fontsize=14)
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.grid(True, alpha=0.3)

# True states
ax2 = axes[1]
ax2.fill_between(range(T), true_states, alpha=0.6, step='mid', color='blue')
ax2.set_ylabel('State', fontsize=12)
ax2.set_title('True Hidden States', fontsize=14)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Bull', 'Bear'])
ax2.grid(True, alpha=0.3)

# Decoded states
ax3 = axes[2]
ax3.fill_between(range(T), decoded_path, alpha=0.6, step='mid', color='orange')
ax3.set_ylabel('State', fontsize=12)
ax3.set_xlabel('Time', fontsize=12)
ax3.set_title(f'Viterbi Decoded States (Accuracy: {accuracy:.1%})', fontsize=14)
ax3.set_yticks([0, 1])
ax3.set_yticklabels(['Bull', 'Bear'])
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Viterbi vs. Forward-Backward

Let's compare Viterbi (most likely path) with forward-backward (marginal posteriors).

In [ ]:
# Purpose: Compare Viterbi path with forward-backward marginals
# Key Concept: Global optimum vs. local marginal probabilities

# For this we need forward-backward (simplified version)
def forward_backward_gaussian(observations, pi, A, means, stds):
    """Simplified forward-backward for Gaussian HMM."""
    T = len(observations)
    K = len(pi)
    
    # Compute emission probabilities
    B = np.zeros((T, K))
    for t in range(T):
        for k in range(K):
            B[t, k] = stats.norm.pdf(observations[t], means[k], stds[k])
    
    # Forward
    alpha = np.zeros((T, K))
    alpha[0] = pi * B[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * B[t]
    
    # Backward
    beta = np.zeros((T, K))
    beta[T-1] = 1.0
    for t in range(T-2, -1, -1):
        beta[t] = A @ (B[t+1] * beta[t+1])
    
    # Gamma
    gamma = alpha * beta
    gamma /= gamma.sum(axis=1, keepdims=True)
    
    return gamma

# Compute forward-backward posteriors
gamma_fb = forward_backward_gaussian(
    observations_cont, true_pi, true_A, true_means, true_stds
)

# Marginal maximization path
marginal_path = np.argmax(gamma_fb, axis=1).tolist()
marginal_accuracy = np.mean(np.array(marginal_path) == np.array(true_states))

print("Comparison: Viterbi vs. Forward-Backward")
print("="*60)
print(f"Viterbi accuracy:   {accuracy:.1%}")
print(f"Marginal accuracy:  {marginal_accuracy:.1%}")
print(f"\nPaths agree:        {np.mean(np.array(decoded_path) == np.array(marginal_path)):.1%} of the time")

# Find disagreements
disagreements = np.where(np.array(decoded_path) != np.array(marginal_path))[0]
print(f"\nNumber of disagreements: {len(disagreements)}")
if len(disagreements) > 0:
    print(f"First few disagreements at times: {disagreements[:5]}")

## Exercise 1: Complete Viterbi Implementation

**Task:** Implement a complete Viterbi function that handles both discrete and continuous (Gaussian) emissions.

**Expected Output:** Function that returns best path and supports both emission types.

In [ ]:
# YOUR CODE HERE
# ---------------

def viterbi_complete(
    observations: np.ndarray,
    pi: np.ndarray,
    A: np.ndarray,
    emission_params: dict
) -> Tuple[List[int], float]:
    """
    Complete Viterbi implementation supporting discrete and Gaussian emissions.
    
    Parameters
    ----------
    emission_params : dict
        Either {'B': emission_matrix} for discrete
        or {'means': means, 'stds': stds} for Gaussian
    
    Returns
    -------
    best_path : list of int
    log_prob : float
    """
    # YOUR IMPLEMENTATION HERE
    
    return None, None

# Test on discrete observations
discrete_params = {'B': B}
path_discrete, prob_discrete = viterbi_complete(observations, pi, A, discrete_params)

# Test on continuous observations
gaussian_params = {'means': true_means, 'stds': true_stds}
path_gaussian, prob_gaussian = viterbi_complete(
    observations_cont, true_pi, true_A, gaussian_params
)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_1():
    assert path_discrete is not None, "Don't forget to implement viterbi_complete!"
    assert path_gaussian is not None, "Gaussian version should work too!"
    
    # Check discrete version
    assert len(path_discrete) == len(observations), "Path length should match observations"
    assert path_discrete == best_path, "Discrete path doesn't match reference"
    
    # Check Gaussian version
    assert len(path_gaussian) == len(observations_cont), "Path length should match observations"
    acc = np.mean(np.array(path_gaussian) == np.array(decoded_path))
    assert acc == 1.0, "Gaussian path doesn't match reference"
    
    print("✅ Exercise 1 passed!")
    print(f"\nDiscrete: {path_discrete}")
    print(f"Gaussian accuracy: {np.mean(np.array(path_gaussian) == np.array(true_states)):.1%}")

test_exercise_1()

<details>
<summary>Hint 1</summary>
Check if 'B' is in emission_params for discrete, otherwise use 'means' and 'stds' for Gaussian.
</details>

<details>
<summary>Hint 2</summary>
For Gaussian, compute log emission probabilities using stats.norm.logpdf before running Viterbi.
</details>

## Exercise 2: Regime Transition Detection

**Task:** Use Viterbi to identify regime transitions in a financial time series.

Given daily returns, find:
1. The most likely state sequence
2. Time points where regime changes occur
3. Average duration of each regime

**Expected Output:** List of transition times and regime statistics.

In [ ]:
# Generate longer market data
np.random.seed(456)
T_long = 500

true_states_long = [0]
for _ in range(T_long-1):
    true_states_long.append(np.random.choice(2, p=true_A[true_states_long[-1]]))

returns_long = np.array([
    np.random.normal(true_means[s], true_stds[s]) for s in true_states_long
])

# YOUR CODE HERE
# ---------------

def analyze_regime_transitions(returns, pi, A, means, stds):
    """
    Analyze regime transitions using Viterbi.
    
    Returns
    -------
    transitions : list of int
        Time indices where regime changed
    durations : dict
        Average duration of each regime
    """
    # YOUR IMPLEMENTATION HERE
    
    return None, None

transitions, durations = analyze_regime_transitions(
    returns_long, true_pi, true_A, true_means, true_stds
)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_2():
    assert transitions is not None, "Don't forget to find transitions!"
    assert durations is not None, "Don't forget to compute durations!"
    
    # Check transitions are valid
    assert all(0 <= t < T_long for t in transitions), "Invalid transition times"
    assert len(transitions) > 0, "Should find at least some transitions"
    
    # Check durations
    assert 0 in durations and 1 in durations, "Should have durations for both states"
    assert all(d > 0 for d in durations.values()), "Durations should be positive"
    
    print("✅ Exercise 2 passed!")
    print(f"\nNumber of regime transitions: {len(transitions)}")
    print(f"Average Bull duration: {durations[0]:.1f} days")
    print(f"Average Bear duration: {durations[1]:.1f} days")
    print(f"\nFirst 5 transitions at days: {transitions[:5]}")

test_exercise_2()

<details>
<summary>Hint 1</summary>
Run viterbi_gaussian to get the state sequence, then find where consecutive states differ.
</details>

<details>
<summary>Hint 2</summary>
For durations, identify contiguous segments of the same state and compute their lengths.
</details>

## Exercise 3: Path Probability Comparison

**Task:** For a given observation sequence, compare:
1. Probability of Viterbi path
2. Probability of marginal maximization path
3. Show that Viterbi path has higher (or equal) probability

**Expected Output:** Probability comparison demonstrating Viterbi optimality.

In [ ]:
# YOUR CODE HERE
# ---------------

def compute_path_probability(
    path: List[int],
    observations: np.ndarray,
    pi: np.ndarray,
    A: np.ndarray,
    means: np.ndarray,
    stds: np.ndarray
) -> float:
    """
    Compute the log probability of a specific state path.
    
    P(path, obs) = π[path[0]] × ∏ₜ A[path[t-1], path[t]] × P(obs[t] | path[t])
    """
    # YOUR IMPLEMENTATION HERE
    
    return None

# Compute probabilities
viterbi_prob = compute_path_probability(
    decoded_path, observations_cont, true_pi, true_A, true_means, true_stds
)

marginal_prob = compute_path_probability(
    marginal_path, observations_cont, true_pi, true_A, true_means, true_stds
)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise_3():
    assert viterbi_prob is not None, "Don't forget to compute viterbi_prob!"
    assert marginal_prob is not None, "Don't forget to compute marginal_prob!"
    
    # Viterbi should have higher or equal probability
    assert viterbi_prob >= marginal_prob - 1e-6, \
        "Viterbi path should have highest probability!"
    
    print("✅ Exercise 3 passed!")
    print(f"\nViterbi path log-prob:   {viterbi_prob:.4f}")
    print(f"Marginal path log-prob:  {marginal_prob:.4f}")
    print(f"Difference:              {viterbi_prob - marginal_prob:.4f}")
    
    if viterbi_prob > marginal_prob:
        print("\n✓ Viterbi path is strictly better than marginal maximization")
    else:
        print("\n✓ Both paths have equal probability (happen to coincide)")

test_exercise_3()

<details>
<summary>Hint 1</summary>
Start with log π[path[0]], then add log transition probabilities and log emission probabilities.
</details>

<details>
<summary>Hint 2</summary>
Use stats.norm.logpdf for emission log probabilities and np.log for transitions.
</details>

## Summary

### Key Takeaways

1. **Viterbi finds the globally optimal state sequence**, not just local marginal maxima
2. **Log-space computation** is essential for numerical stability
3. **Backtracking** recovers the optimal path after forward pass
4. **Gaussian HMMs** use probability densities instead of discrete emission matrices
5. **Viterbi ≥ Marginal** in probability - optimal path has highest joint probability

### Algorithm Comparison

| Algorithm | Question | Output |
|-----------|----------|--------|
| Forward | P(O\|λ)? | Likelihood |
| Forward-Backward | P(qₜ=i\|O)? | State posteriors |
| Viterbi | Best path? | Most likely sequence |

### When to Use Each

- **Forward-Backward**: When you want probability distributions over states (uncertainty quantification)
- **Viterbi**: When you need a single best guess for the state sequence (hard decoding)

### What's Next

In the next notebook, we'll implement the **Baum-Welch algorithm** to learn HMM parameters from data.

---

**Next Notebook:** `03_em_training.ipynb` - Learning HMM parameters with Expectation-Maximization